# Комбинация: Supervisor + Debate — инвестиционный анализ

Супервайзер координирует три этапа: сбор данных, дебаты аналитиков, подготовку отчёта. Этап дебатов — это **подграф**, где бык (оптимист) и медведь (пессимист) спорят 2 раунда, а затем медиатор выносит взвешенный вердикт. Супервайзер не знает о внутренней структуре дебатов — он отправляет задачу в `debate_team` как в обычный узел и получает только итоговый вердикт.

```mermaid
graph LR
    START((START)) --> supervisor{{"🎯 Supervisor<br/><i>маршрутизирует</i>"}}
    supervisor -->|researcher| researcher(["🔹 Researcher<br/><i>собирает данные</i>"])
    supervisor -->|debate_team| debate_team
    supervisor -->|reporter| reporter(["🔹 Reporter<br/><i>собирает данные</i>"])
    supervisor -->|FINISH| END((END))
    researcher --> supervisor
    reporter --> supervisor

    subgraph debate_team ["Подграф: Debate Team"]
        bull(["🔹 Bull<br/><i>аргументирует</i>"]) --> bear(["🔹 Bear<br/><i>аргументирует</i>"])
        bear -->|раунд < 2| bull
        bear -->|раунд >= 2| mediator(["📊 Mediator<br/><i>выносит вердикт</i>"])
    end

    debate_team --> supervisor

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef subgraphStyle fill:#FDEBD0,stroke:#E67E22,stroke-width:2px,color:#333

    class START,END terminal
    class supervisor coordinator
    class researcher,reporter worker
    class bull,bear worker
    class mediator aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


In [3]:
llm = get_llm()

## Подграф: Debate Team

Дебаты компилируются как отдельный граф и затем встраиваются в основной граф супервайзера как единый узел. Состояние `DebateState` содержит аргументы быка и медведя с редьюсером `operator.add` — каждый раунд добавляет новый аргумент в список, не перезаписывая предыдущие. Поле `round` отсчитывает раунды, а `verdict` заполняется медиатором после завершения дебатов.

In [4]:
class DebateState(TypedDict):
    topic: str
    bull_args: Annotated[list[str], operator.add]
    bear_args: Annotated[list[str], operator.add]
    round: int
    verdict: str

## Узлы дебатов: бык, медведь, медиатор

Агент-бык фокусируется на возможностях и позитивных сценариях, агент-медведь — на рисках и угрозах. Каждый получает аргументы оппонента из предыдущего раунда, что обеспечивает реальную дискуссию, а не параллельные монологи. Медиатор синтезирует обе позиции в сбалансированное заключение после завершения всех раундов.

In [5]:
def bull_node(state: DebateState) -> dict:
    opponent = "\n".join(state["bear_args"]) if state["bear_args"] else "Нет аргументов."
    response = llm.invoke([
        SystemMessage(content="Ты — оптимист-аналитик (бык). Найди возможности и позитивные сценарии. 2-3 аргумента."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nАргументы медведя:\n{opponent}")
    ])
    r = state.get("round", 0) + 1
    print(f"  [Бык, раунд {r}] {response.content[:60]}...")
    return {"bull_args": [f"Раунд {r}: {response.content}"], "round": r}


def bear_node(state: DebateState) -> dict:
    opponent = "\n".join(state["bull_args"])
    response = llm.invoke([
        SystemMessage(content="Ты — пессимист-аналитик (медведь). Найди риски и негативные сценарии. 2-3 аргумента."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nАргументы быка:\n{opponent}")
    ])
    print(f"  [Медведь, раунд {state['round']}] {response.content[:60]}...")
    return {"bear_args": [f"Раунд {state['round']}: {response.content}"]}


def mediator_node(state: DebateState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Ты — медиатор. Синтезируй сбалансированное заключение из дебатов. 3-4 предложения."),
        HumanMessage(content=(
            f"Тема: {state['topic']}\n\n"
            f"Бык:\n" + "\n".join(state["bull_args"]) + "\n\n"
            f"Медведь:\n" + "\n".join(state["bear_args"])
        ))
    ])
    print(f"  [Медиатор] Вердикт готов")
    return {"verdict": response.content}

## Сборка подграфа дебатов

Маршрутизация внутри подграфа: `bull -> bear -> (условное ребро)`. Если прошло меньше 2 раундов — возврат к быку для продолжения дебатов. Если 2 раунда завершены — управление переходит к медиатору, который выносит финальный вердикт. Скомпилированный подграф `debate_compiled` станет обычным узлом в основном графе.

In [6]:
def debate_continue(state: DebateState) -> str:
    return "mediate" if state["round"] >= 2 else "debate"

debate_graph = StateGraph(DebateState)
debate_graph.add_node("bull", bull_node)
debate_graph.add_node("bear", bear_node)
debate_graph.add_node("mediator", mediator_node)
debate_graph.add_edge(START, "bull")
debate_graph.add_edge("bull", "bear")
debate_graph.add_conditional_edges("bear", debate_continue, {"debate": "bull", "mediate": "mediator"})
debate_graph.add_edge("mediator", END)
debate_compiled = debate_graph.compile()

## Основной граф: состояние супервайзера

Состояние `MainState` содержит данные, которые передаются между этапами анализа. Поле `next_agent` — способ супервайзера сообщить маршрутизатору, куда направить управление дальше. Обратите внимание: поля `topic` и `verdict` совпадают по именам с полями `DebateState` — это позволяет подграфу автоматически получать тему и возвращать вердикт.

In [7]:
class MainState(TypedDict):
    topic: str
    research: str
    verdict: str
    report: str
    next_agent: str

## Рабочие узлы: исследователь и репортёр

Исследователь собирает ключевые факты по теме. Репортёр создаёт итоговый отчёт на основе фактов и вердикта дебатов. Оба агента — простые узлы с одним вызовом LLM, без внутренней сложности.

In [8]:
def researcher_node(state: MainState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Ты — исследователь. Собери 3-4 ключевых факта по теме. Кратко."),
        HumanMessage(content=state["topic"])
    ])
    print(f"  [Исследователь] Факты собраны")
    return {"research": response.content}


def reporter_node(state: MainState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Ты — аналитик. Напиши итоговый отчёт (4-5 предложений) на основе фактов и вердикта дебатов."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nФакты:\n{state['research']}\n\nВердикт дебатов:\n{state['verdict']}")
    ])
    print(f"  [Репортёр] Отчёт готов")
    return {"report": response.content}

## Узел супервайзера

Супервайзер анализирует текущий статус работы и решает, какого агента вызвать следующим. Он получает от LLM JSON вида `{"next": "agent_name"}` и передаёт имя в `next_agent`. Порядок: `researcher -> debate_team -> reporter -> FINISH`. Критически важен fallback при парсинге JSON: LLM не всегда отвечает чистым JSON, поэтому при ошибке парсинга мы ищем имя агента в тексте ответа.

In [9]:
def supervisor_node(state: MainState) -> dict:
    status = []
    if state.get("research"): status.append("Исследование: готово")
    else: status.append("Исследование: нет")
    if state.get("verdict"): status.append("Дебаты: завершены")
    elif state.get("research"): status.append("Дебаты: не проведены")
    if state.get("report"): status.append("Отчёт: готов")

    response = llm.invoke([
        SystemMessage(content="""Ты — супервайзер. Команда: researcher, debate_team, reporter.
Порядок: researcher → debate_team → reporter → FINISH.
Верни JSON: {"next": "researcher"} / {"next": "debate_team"} / {"next": "reporter"} / {"next": "FINISH"}"""),
        HumanMessage(content=f"Тема: {state['topic']}\n\nСтатус:\n" + "\n".join(status))
    ])
    try:
        next_agent = json.loads(response.content.strip()).get("next", "FINISH")
    except json.JSONDecodeError:
        c = response.content.lower()
        if "research" in c: next_agent = "researcher"
        elif "debate" in c: next_agent = "debate_team"
        elif "report" in c: next_agent = "reporter"
        else: next_agent = "FINISH"
    print(f"  [Супервайзер] → {next_agent}")
    return {"next_agent": next_agent}

## Маршрутизация и сборка основного графа

Условное ребро из супервайзера направляет управление к нужному агенту на основе `next_agent`. Ключевой момент — `debate_compiled` (скомпилированный подграф дебатов) регистрируется как обычный узел через `graph.add_node("debate_team", debate_compiled)`. LangGraph автоматически передаёт совпадающие по именам поля между `MainState` и `DebateState`. После каждого рабочего узла — безусловное ребро обратно к супервайзеру, формируя цикл.

In [10]:
def route(state: MainState) -> str:
    n = state.get("next_agent", "FINISH")
    return END if n == "FINISH" else n

graph = StateGraph(MainState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("debate_team", debate_compiled)  # подграф дебатов как узел!
graph.add_node("reporter", reporter_node)

graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route, {
    "researcher": "researcher", "debate_team": "debate_team",
    "reporter": "reporter", END: END,
})
graph.add_edge("researcher", "supervisor")
graph.add_edge("debate_team", "supervisor")
graph.add_edge("reporter", "supervisor")

app = graph.compile()

## Запуск

Типичный порядок работы: `supervisor -> researcher -> supervisor -> debate_team (bull <-> bear -> mediator) -> supervisor -> reporter -> supervisor -> FINISH`. Супервайзер вызывается между каждым этапом и принимает решение о следующем шаге на основе текущего статуса.

In [11]:
result = app.invoke({
    "topic": "Перспективы инвестиций в AI-стартапы в 2026 году",
    "research": "", "verdict": "", "report": "", "next_agent": "",
})

print(f"\n{'=' * 60}")
print(f"ИТОГОВЫЙ ОТЧЁТ:")
print(f"{'=' * 60}")
print(result.get("report", "Нет")[:500])

  [Супервайзер] → researcher


  [Исследователь] Факты собраны


  [Супервайзер] → debate_team


  [Бык, раунд 1] Вот бычий взгляд на перспективы инвестиций в AI-стартапы в 2...


  [Медведь, раунд 1] Пессимистичный взгляд: в 2026 году AI-стартапы останутся при...


...

  [Медведь, раунд 2] С медвежьей позиции я бы смотрел на AI-стартапы в 2026 году ...


  [Медиатор] Вердикт готов


  [Супервайзер] → reporter


  [Репортёр] Отчёт готов


  [Супервайзер] → FINISH

ИТОГОВЫЙ ОТЧЁТ:
В 2026 году инвестиции в AI-стартапы сохранят привлекательность, но рынок станет заметно более избирательным: капитал будет уходить не в «AI ради AI», а в компании с понятной выручкой, сильной unit-экономикой и защищаемым преимуществом. Наиболее перспективными выглядят прикладные B2B-решения для автоматизации продаж, поддержки, аналитики, разработки, медицины, финтеха и кибербезопасности, а также инфраструктурные и нишевые модели. При этом риски высокой конкуренции, зависимости от крупных платфо


## Итоги

**Supervisor + Debate** демонстрирует ключевой принцип композиции паттернов: подграф дебатов встраивается в основной граф как обычный узел. Супервайзер не знает, что внутри `debate_team` происходят 2 раунда дебатов между быком и медведем с последующим вердиктом медиатора — он просто отправляет задачу и получает результат.

Преимущества комбинации:
- **Взвешенность**: вместо однобокого прогноза — синтез оптимистичной и пессимистичной позиций
- **Модульность**: подграф дебатов можно заменить, усложнить или переиспользовать в других системах
- **Контролируемость**: супервайзер управляет порядком этапов, а подграф — внутренней логикой дебатов